# Stage 1.12–1.13 — Coffee Modeling Table & Splits

Turn the coffee competitive-set reviews into a clean table (text + label),
then split by time for training.

## Libraries

In [4]:
import pandas as pd
from pathlib import Path

## 1.12 Build the modeling table

In [6]:
ROOT = Path.cwd().parents[0] 
df = pd.read_parquet(ROOT / "data/processed/coffee_reviews_cset.parquet")
before = len(df)

df["date"] = pd.to_datetime(df["timestamp"], unit="ms")

df["full_text"] = (
    df["title"].fillna("") + ". " + df["text"].fillna("")
).str.strip()

df["label"] = df["rating"].map({1: 0, 2: 0, 4: 1, 5: 1})

df = df[df["label"].notna()]
df = df[df["full_text"].str.len() >= 20].copy()
df["label"] = df["label"].astype(int)

print(f"before: {before}  after: {len(df)}")
print(df["label"].value_counts(normalize=True).round(3))

before: 1237303  after: 1125994
label
1    0.845
0    0.155
Name: proportion, dtype: float64


## 1.13 Time-based train/val/test split

In [8]:
df = df.sort_values("date")

n = len(df)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

train = df.iloc[:train_end]
val   = df.iloc[train_end:val_end]
test  = df.iloc[val_end:]

train.to_parquet(ROOT / "data/processed/coffee_train.parquet")
val.to_parquet(ROOT / "data/processed/coffee_val.parquet")
test.to_parquet(ROOT / "data/processed/coffee_test.parquet")

print(f"train {len(train)}  ({train['date'].min().date()} → {train['date'].max().date()})")
print(f"val   {len(val)}  ({val['date'].min().date()} → {val['date'].max().date()})")
print(f"test  {len(test)}  ({test['date'].min().date()} → {test['date'].max().date()})")

train 788195  (2003-09-25 → 2021-02-01)
val   168899  (2021-02-01 → 2022-02-25)
test  168900  (2022-02-25 → 2023-09-12)
